# rlmflow coding-agent walkthrough

Build a recursive coding agent, run a task, save the run, and embed the generated artifact inline. This mirrors `examples/coding/agent.py` but as a notebook so you can poke at every step.

Sibling notebooks read offline against a saved run directory under `examples/_runs/word-search/baseline/`:

- [`node_basics.ipynb`](./node_basics.ipynb) — querying a run with `walk`, `find`, `transcript`, and `persistence.load`
- [`viz_walkthrough.ipynb`](./viz_walkthrough.ipynb) — visualizing a run: `tree`, `gantt`, `code_log`, mermaid / dot / d2, `report_md`, the inline Gradio viewer, etc.

Running this notebook end-to-end requires `OPENAI_API_KEY` and live LLM calls. Skip ahead to the other notebooks if you just want to consume a saved run.

## 1. Build the agent

The whole run is one `Flow` driving one root `Node`:

- `Flow` — wires an LLM client (plus optional cheaper alternates registered as `fast`) to a stateful REPL. `flow.start(query)` seeds the run with that flow's defaults; `async for node in flow.run_streaming(root)` then drives the whole tree, mutating that root in place and yielding each node as it lands. (`flow.run(query)` / `await flow.arun(query)` wrap this when you only want the final result.)
- `Runtime` — *where* code runs. `LocalRuntime` uses an isolated Python worker process. Tools go on the flow: `Flow(tools=FILE_TOOLS)` puts the filesystem tools (`read_file`, `write_file`, `edit_file`, `ls`, `grep`, …) in every agent's REPL and describes them in its system prompt — no `Flow` subclass.
- `working_directory` — set it on the runtime and agent code (and the file tools) run inside it; no manual `chdir`. Swap `LocalRuntime` for `DockerRuntime(image, working_directory=...)` to use a container with the same interface.

In [27]:
from pathlib import Path
import shutil
from rlmflow.llm import OpenAIClient
from rlmflow import AgentConfig, FILE_TOOLS, Flow, LocalRuntime


def build_agent(
    workdir: Path | str,
    max_depth: int = 2,
    max_iters: int = 30,
    workers: int = 8,
) -> Flow:
    """Construct a coding agent identical to examples/coding/agent.py."""
    # LocalRuntime starts an isolated Python worker in workdir. Swap in
    # DockerRuntime to use a container. Tools go on the flow, which seeds them
    # into every agent's REPL and describes them in the system prompt.
    return Flow(
        OpenAIClient("gpt-5"),
        llm_clients={"fast": OpenAIClient("gpt-5-mini")},
        runtime=LocalRuntime(working_directory=workdir),
        tools=FILE_TOOLS,
        # Per-agent limits live on AgentConfig; these are the flow's defaults for
        # roots and children. `workers` caps how many blocking calls run at once.
        root_config=AgentConfig(max_depth=max_depth, max_iters=max_iters),
        workers=workers,
    )

## 2. Run a task

Calling `agent.start(query)` builds the root `AgentStart`, carrying the limits the flow was built with. `agent.run_streaming(root)` then drives the whole tree, mutating that same root in place and yielding one `Node` per step — one model call, one REPL execution, one answer. Use `root.frontier` to inspect the latest node, `root.terminal` to check for completion, and `root.result()` for the answer.

Nothing is hidden behind a callback: the loop below is the whole thing. Each yielded node is drawn into one IPython display handle (so the tree animates mid-cell) and the root is saved to disk on every step, so a crash still leaves the latest tree on disk. The node tree stays the durable source of truth.


In [28]:
TASK = """Create a runnable browser-based boids simulation in plain HTML, CSS, and JavaScript. They should be fast moving, colorful, and fun.

Delegate subagents for the three artifacts (index.html, style.css, boids.js)
so each file is owned by its own child; as root, wire them together and verify.
No build tools, no libraries, no ES modules.

Requirements:
- index.html — canvas + <script src="..."> tags (no inline JS/CSS)
- style.css — dark background; canvas fills the viewport
- boids.js — hundreds of colorful boids on a 2D canvas; no config UI
- Verify files exist and script tags are ordered correctly before done(...)
"""

WORKDIR = Path("../_runs/notebooks/boids-sim").resolve()
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
WORKDIR.mkdir(parents=True, exist_ok=True)

agent = build_agent(WORKDIR, max_depth=2)

In [29]:
graph = agent.start(TASK)
# Built from the live prompt builder so the preview tracks prompt changes.
prompt = agent.system_prompt.render(agent, graph)
print(prompt)

You are a Recursive Coding Agent: a language model with a user query and important
inputs stored in a Python REPL. You are queried turn-by-turn until you have an
answer. To use the REPL, write code in ```repl``` blocks; it persists across turns.

Available in the REPL:

1. `INPUTS`: a dict of string inputs (may be empty). Your task arrives as the user
   message, not in `INPUTS`. Keys are caller-defined — inspect `list(INPUTS)`
   rather than assuming names; parse JSON with `json.loads(INPUTS["key"])`. Keys
   never shadow REPL variables or tools.
2. `await launch_subagents(specs) -> list`: recursive sub-agent calls. Each spec
   needs a short `query` and may set `inputs` (str -> str), `name`, `model`, and
   `output_schema`. Keep `query` a one/two-sentence instruction and put large
   payloads in `inputs`. Returns a list even for one child.
3. `print(...)`: only stdout is shown back between turns; a bare final expression
   is discarded. Never dump large `INPUTS` values — REPL output 

In [31]:
import asyncio

from IPython.display import display


def render_tree(node, depth=0):
    """One line per node, indented by depth. The tree is the whole state."""
    label = node.config.path if node.type == "agent_start" else node.type
    lines = [f"{'  ' * depth}{label}"]
    for child in node.children:
        lines.extend(render_tree(child, depth + 1))
    return lines


# `run_streaming` mutates `graph` in place and yields each node as it lands.
# Keep one display handle and `.update(...)` it so the tree animates live
# (clear_output + print only shows the final frame once the cell ends).
handle = display("\n".join(render_tree(graph)), display_id=True)

async for node in agent.run_streaming(graph):
    handle.update("\n".join(render_tree(graph)))
    graph.save(WORKDIR / "graph")  # checkpoint every step
    await asyncio.sleep(0)  # yield so the frontend can paint mid-run

In [32]:
# Actual rlm run

In [22]:
spent = graph.tokens()
print(f"{sum(1 for _ in graph.walk())} nodes  ·  frontier: {graph.frontier.type}")
print(f"tokens: {spent.input_tokens} in / {spent.output_tokens} out")
print(f"query : {graph.content[:120]!r}...")
print(f"result: {str(graph.result())[:200]}")

45 snapshots  ·  final: root [done_output]
query : 'Create a runnable browser-based boids simulation in plain HTML, CSS, and JavaScript. They should be fast moving, colorfu'...
result: Created files but verification failed. See console for details on what to fix: boids.js: Must get canvas by id 'boids'.; boids.js: Must use 2D canvas context.; boids.js: Missing requestAnimationFrame 


In [34]:
print("\n".join(render_tree(graph)))

Create a runnable browser-based boids simulation in plain HTML, CSS, and JavaSc...
└── root: done Boids simulation is ready. Open index.html in a browser to run ... (2 turns)
    ├── root.index.html author: done {"content": "<!doctype html>\n<html lang=\"en\">\n<head>\n <met... (1 turns)
    ├── root.style.css author: done {"content": "html,body{height:100%;width:100%;margin:0;padding:... (1 turns)
    └── root.boids.js author: done {"content": "(function () {\n 'use strict';\n\n // Fast, colorf... (1 turns)


In [35]:
# The loop above already saved on every step; this is the same call — a run
# directory (manifest + nested per-agent logs) alongside the files the agent
# wrote into WORKDIR. Reload any run dir with `AgentStart.load(path)`.
run_dir = graph.save(WORKDIR / "graph")
agents = sum(1 for node in graph.walk() if node.type == "agent_start")
print(f"run -> {run_dir}  ({agents} agents)")

run -> /Users/shyam/Code/rlmkit/examples/_runs/notebooks/boids-sim/graph  (4 agents)


## 3. Preview the generated artifact

Serve the generated `index.html` over local HTTP and embed that URL in the notebook. This exercises the same browser module-loading rules as opening the artifact normally, so import/export mistakes are not hidden by `srcdoc` inlining.

In [36]:
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from IPython.display import IFrame
import socket
import threading

html_path = (WORKDIR / "index.html").resolve()
if not html_path.is_file():
    candidates = sorted(p for p in WORKDIR.glob("**/index.html") if "graph" not in p.parts)
    if not candidates:
        raise FileNotFoundError(f"no index.html under {WORKDIR}")
    html_path = candidates[0].resolve()
site_root = html_path.parent

_prev = globals().get("preview_server")
if _prev is not None:
    _prev.shutdown()
    _prev.server_close()

with socket.socket() as s:
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]


class QuietHandler(SimpleHTTPRequestHandler):
    def log_message(self, *args):
        pass


handler = partial(QuietHandler, directory=str(site_root))
preview_server = ThreadingHTTPServer(("127.0.0.1", port), handler)
threading.Thread(target=preview_server.serve_forever, daemon=True).start()

url = f"http://127.0.0.1:{port}/{html_path.name}"
print(f"serving {site_root} -> {url}")
display(IFrame(url, width="100%", height=600))

serving /Users/shyam/Code/rlmkit/examples/_runs/notebooks/boids-sim -> http://127.0.0.1:63358/index.html


## 4. Read the run back

`AgentStart.load(path)` rebuilds the tree the loop above saved — the same object, ids and all. Each agent's own turns are `agent.transcript()`; the children it launched are `agent.sub_agents`.

In [37]:
from rlmflow import AgentStart

loaded = AgentStart.load(WORKDIR / "graph")
for agent in [loaded, *loaded.sub_agents]:
    turns = [node.type for node in agent.transcript()]
    print(f"{agent.config.path:<16} {len(turns):>3} nodes  {' '.join(turns[:6])} ...")

## 5. Ask for one more thing

A finished run is not a closed one. Append a `UserQuery` to the frontier and stream it again: the transcript and the warm REPL are still there, so the agent picks up where it left off rather than re-deriving its own work.

In [ ]:
from rlmflow import UserQuery

graph.frontier.append(UserQuery(content="Add a slider for the flock's cohesion."))

async for node in agent.run_streaming(graph):
    handle.update("\n".join(render_tree(graph)))
    await asyncio.sleep(0)

print(str(graph.result())[:200])

## Next

- [`examples/graph/`](../graph/) — the node API on its own: query, navigate, save/load, fork.
- [`examples/coding/agent.py`](../coding/agent.py) — the same agent as a script, with Docker as an option.